# Phase 9: Final Evaluation, Error Analysis and Deployment (Q49–Q50)

**Questions covered:** Q49 (full evaluation on held-out data, error analysis, sample predictions) and Q50 (working Streamlit app that loads the saved model, shows the class, confidence and probabilities, and handles bad files).

## The most important rule of this notebook
The **test split has not been used for anything so far.** It is touched here **once**, in this order:

1. **Choose the final model using VALIDATION results only** (cell 4). Write the name down. *Do not change it after you see test numbers.*
2. Evaluate **every** model on the test set, for reporting only (it is not used to choose anything).
3. Analyse the errors of the chosen model.
4. Package the chosen model and test the app code.

If the test numbers are disappointing, report them honestly. Never go back and re-select a model because of them.

**Deployment plan:** the Streamlit app runs on Colab (the pod is not a good place to host it) and, as a fallback, on your own laptop. The model travels as a `state_dict` plus a JSON config in one zip (`app_package.zip`), not as a pickled model, so it loads across PyTorch versions.

In [ ]:
import os
for var in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"]:
    os.environ[var] = "8"
import sys, json, time, random, importlib, shutil
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, torch, joblib
from pathlib import Path
from sklearn.metrics import classification_report, roc_curve

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.set_num_threads(8)
PROJECT_ROOT = Path("/workspace/sic_final_project")      # change here if the folder moves
sys.path.insert(0, str(PROJECT_ROOT / "src"))
CNN_DIR, TL_DIR, TRAD_DIR = PROJECT_ROOT / "models/cnn", PROJECT_ROOT / "models/transfer", PROJECT_ROOT / "models/traditional"
WEIGHTS_DIR = PROJECT_ROOT / "models/pretrained"
for d in ["results/figures", "results/metrics", "results/comparisons", "app", "reports"]:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

essential = ["data/splits/split_seed42.csv", "data/metadata/metadata_clean.csv", "data/processed/images_64.npy",
             "data/processed/images_128.npy", "data/processed/images_224.npy", "data/processed/labels.npy",
             "results/comparisons/model_results.csv", "results/metrics/05_traditional_configs.json",
             "src/evaluation.py", "src/traditional_models.py", "src/gpu_models.py", "src/cnn_models.py",
             "src/training.py", "src/pretrained_models.py"]
optional = ["results/metrics/06_feature_configs.json", "models/traditional/pca_rgb64.pt", "data/processed/hog_gray128.npy",
            "models/traditional/logreg_raw64_gpu.pt", "models/traditional/random_forest_raw64.joblib",
            "models/cnn/best_custom_cnn.json", "models/cnn/cnn_aug.pt", "models/transfer/best_transfer.json"]
miss = [f for f in essential if not (PROJECT_ROOT / f).exists()]
for f in essential + optional:
    print(("OK       " if (PROJECT_ROOT / f).exists() else ("MISSING  " if f in essential else "optional-missing ")), f)
assert not miss, "Essential files are missing: finish the earlier phases first."

## Step 1: `prediction.py` (shared by the notebook tests and the app)

It loads the image, converts it to RGB, resizes it **exactly as in Phase 4** (direct resize to a square, PIL bilinear), normalises it with the **training** statistics from the config, and returns the class, the two probabilities and the confidence. Anything that is not a readable image (or is too small) raises `InvalidImageError`, which the app turns into a friendly message.

In [ ]:
%%writefile /workspace/sic_final_project/src/prediction.py
"""Single-image inference for the saved Cat-vs-Dog model. Used by the Streamlit app and by the notebooks."""
import io
import json
import sys
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image

_HERE = Path(__file__).resolve().parent
if str(_HERE) not in sys.path:
    sys.path.insert(0, str(_HERE))

CLASS_NAMES = ["Cat", "Dog"]          # label 0 = Cat, label 1 = Dog (Dog is the positive class)
MIN_SIDE = 32                         # the same limit as the cleaning step (Phase 3)
MAX_PIXELS = 100_000_000


class InvalidImageError(ValueError):
    """The input is not a usable image."""


def load_image(source):
    """source: file path, bytes, or a file-like object. Returns an RGB PIL image or raises InvalidImageError."""
    try:
        if isinstance(source, (bytes, bytearray)):
            source = io.BytesIO(bytes(source))
        img = Image.open(source)
        if img.width * img.height > MAX_PIXELS:
            raise InvalidImageError("the image is too large")
        img.load()
    except InvalidImageError:
        raise
    except Exception as exc:
        raise InvalidImageError("the file is not a readable image (supported: JPG, PNG, BMP, GIF, WebP)") from exc
    if min(img.size) < MIN_SIDE:
        raise InvalidImageError(f"the image is too small (shortest side must be at least {MIN_SIDE} pixels)")
    return img.convert("RGB")


class Predictor:
    def __init__(self, model, cfg, device):
        self.model, self.cfg, self.device = model, cfg, device
        self.size = int(cfg["input_size"])
        self.mean = torch.tensor(cfg["mean"], dtype=torch.float32).view(1, 3, 1, 1)
        self.std = torch.tensor(cfg["std"], dtype=torch.float32).view(1, 3, 1, 1)
        self.threshold = float(cfg.get("threshold", 0.5))

    def preprocess(self, rgb):
        arr = np.asarray(rgb.resize((self.size, self.size), Image.BILINEAR), dtype=np.uint8)
        x = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).float() / 255.0
        return ((x - self.mean) / self.std).to(self.device)

    @torch.no_grad()
    def predict(self, source):
        t0 = time.perf_counter()
        rgb = load_image(source)
        p_dog = float(torch.sigmoid(self.model(self.preprocess(rgb)).float()).reshape(-1)[0])
        label = CLASS_NAMES[int(p_dog >= self.threshold)]
        return {"label": label, "p_dog": p_dog, "p_cat": 1.0 - p_dog,
                "confidence": max(p_dog, 1.0 - p_dog), "ms": 1000 * (time.perf_counter() - t0)}


def load_predictor(folder, device=None):
    """folder contains model.pt (state_dict) and model_config.json."""
    folder = Path(folder)
    cfg = json.loads((folder / "model_config.json").read_text())
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if cfg["family"] == "custom_cnn":
        from cnn_models import CustomCNN
        model = CustomCNN(**cfg["arch_kwargs"], input_size=cfg["input_size"])
    else:
        from pretrained_models import TransferModel
        model = TransferModel(cfg["backbone"], weights_path=None, dropout=cfg.get("dropout", 0.3))
    model.load_state_dict(torch.load(folder / "model.pt", map_location="cpu", weights_only=True))
    return Predictor(model.eval().to(device), cfg, device)

## Step 2: the Streamlit app (`app/streamlit_app.py`)

Features: upload (JPG/PNG/BMP/GIF/WebP, max 10 MB), image preview, predicted class, confidence, both probabilities, an **uncertainty threshold** slider (below it the app says "uncertain" instead of guessing), and clear error messages for invalid files. The model is loaded **once** (`st.cache_resource`), never retrained.

In [ ]:
%%writefile /workspace/sic_final_project/app/streamlit_app.py
import sys
from pathlib import Path

import streamlit as st

HERE = Path(__file__).resolve().parent
for p in (HERE, HERE.parent / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
import prediction

PACKAGE = HERE if (HERE / "model.pt").exists() else HERE.parent / "app_package"
ALLOWED = ["jpg", "jpeg", "png", "bmp", "gif", "webp"]
MAX_MB = 10

st.set_page_config(page_title="Cat vs Dog Classifier", page_icon="🐾", layout="centered")


@st.cache_resource(show_spinner="Loading the trained model...")
def get_predictor():
    return prediction.load_predictor(PACKAGE)


def show_image(img, caption):
    try:
        st.image(img, caption=caption, use_container_width=True)
    except TypeError:                                   # older Streamlit versions
        st.image(img, caption=caption, use_column_width=True)


try:
    predictor = get_predictor()
except Exception as exc:
    st.error(f"The model could not be loaded from '{PACKAGE}'. Details: {exc}")
    st.stop()

cfg = predictor.cfg
st.title("Cat vs Dog Image Classifier")
st.caption("Upload a photo and the trained network will say whether it shows a cat or a dog.")

with st.sidebar:
    st.header("Settings")
    uncertain_below = st.slider("Uncertainty threshold", 0.50, 0.99, 0.70, 0.01,
                                help="If the model's confidence is below this value, the app reports 'uncertain' instead of a label.")
    st.header("About the model")
    st.write(f"**{cfg.get('final_model_name', 'trained model')}**")
    st.write(f"Input size: {cfg['input_size']}x{cfg['input_size']}")
    if "test" in cfg:
        st.write(f"Held-out test accuracy: {cfg['test']['accuracy']:.2%}")
    st.write(f"Running on: {predictor.device}")

upload = st.file_uploader("Upload an image", type=ALLOWED)
if upload is None:
    st.info("Choose a JPG, PNG, BMP, GIF or WebP image to classify.")
    st.stop()

data = upload.getvalue()
if len(data) > MAX_MB * 1024 * 1024:
    st.error(f"The file is larger than {MAX_MB} MB. Please upload a smaller image.")
    st.stop()

try:
    result = predictor.predict(data)
    preview = prediction.load_image(data)
except prediction.InvalidImageError as exc:
    st.error(f"This file cannot be classified: {exc}.")
    st.stop()
except Exception:
    st.error("Something went wrong while classifying this image. Please try another file.")
    st.stop()

left, right = st.columns(2)
with left:
    show_image(preview, upload.name)
with right:
    if result["confidence"] < uncertain_below:
        st.warning(f"Uncertain (confidence {result['confidence']:.1%} is below {uncertain_below:.0%}). "
                   f"Leaning towards: {result['label']}.")
    else:
        st.success(f"Prediction: {result['label']}")
    st.metric("Confidence", f"{result['confidence']:.1%}")
    st.write("Class probabilities")
    st.progress(result["p_cat"], text=f"Cat: {result['p_cat']:.1%}")
    st.progress(result["p_dog"], text=f"Dog: {result['p_dog']:.1%}")

with st.expander("Details"):
    st.write(f"Inference time (including preprocessing): {result['ms']:.1f} ms on {predictor.device}")
    st.write("The probabilities come from the network's sigmoid output; they show the model's confidence, "
             "not a guarantee that it is right.")

## Step 3: choose the final model (VALIDATION data only)

Candidates are the deep models (custom CNNs, CNN + augmentation, transfer learning). The no-augmentation *control* run from Phase 8 is excluded because it exists only for the Q47 comparison.

**Rule (written before any test number is seen):**
1. Find the best validation accuracy.
2. Keep every candidate within **0.3 percentage points** of it (differences that small are within noise for a validation set of about 3,700 images).
3. Among those, choose the one with the **fewest parameters** (smaller = faster and cheaper to deploy); if still tied, the higher validation AUC.

You may override the choice with `FINAL_OVERRIDE`, but only for reasons you can justify from validation results, size or speed, and only **before** the test evaluation.

In [ ]:
import evaluation, traditional_models, gpu_models, cnn_models, training, pretrained_models
for m in (evaluation, traditional_models, gpu_models, cnn_models, training, pretrained_models):
    importlib.reload(m)

res = pd.read_csv(evaluation.RESULTS_CSV)
val = res[res["split"] == "val"].copy()
is_deep = val["experiment"].str.startswith(("Custom CNN - ", "CNN + Augmentation", "TL "))
deep = val[is_deep & ~val["experiment"].str.contains("control", case=False)].copy()
deep["train-val gap"] = deep["train_accuracy"] - deep["accuracy"]
cols = ["experiment", "accuracy", "f1", "roc_auc", "params", "infer_ms_per_img", "train_time_s", "train-val gap"]
display(deep.sort_values("accuracy", ascending=False)[cols].round(4).reset_index(drop=True))

best_acc = deep["accuracy"].max()
cand = deep[deep["accuracy"] >= best_acc - 0.003].sort_values(["params", "roc_auc"], ascending=[True, False])
print(f"\nBest validation accuracy: {best_acc:.4f}. Candidates within 0.3 pp: {list(cand['experiment'])}")
FINAL_OVERRIDE = None                     # e.g. "TL ResNet50 (fine-tuned)"; leave None to follow the rule
FINAL_NAME = FINAL_OVERRIDE or cand.iloc[0]["experiment"]
print("\n>>> FINAL MODEL (chosen on validation data):", FINAL_NAME)
print(">>> Write this down. It must not change after the test results appear.")

## Step 4: load the test split (first and only time)

In [ ]:
split_df = pd.read_csv(PROJECT_ROOT / "data/splits/split_seed42.csv")
meta = pd.read_csv(PROJECT_ROOT / "data/metadata/metadata_clean.csv")
labels = np.load(PROJECT_ROOT / "data/processed/labels.npy")
tr = split_df.loc[split_df["split"] == "train", "array_index"].to_numpy()
te = split_df.loc[split_df["split"] == "test", "array_index"].to_numpy()
y_train, y_test = labels[tr], labels[te]

imgs = {s: np.load(PROJECT_ROOT / f"data/processed/images_{s}.npy", mmap_mode="r") for s in (64, 128, 224)}
test_imgs = {s: np.ascontiguousarray(imgs[s][te]) for s in (64, 128, 224)}          # uint8, test split only
print("test images:", len(te), "| classes:", np.bincount(y_test), "| shapes:", {s: a.shape for s, a in test_imgs.items()})

test_df = split_df[split_df["split"] == "test"].merge(meta[["path", "width", "height", "mode", "pil_format"]], on="path", how="left").reset_index(drop=True)
assert len(test_df) == len(te) and (test_df["label_id"].to_numpy() == y_test).all()
TEST = {}            # model name -> dict(metrics, pred, probs, ms)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev)

## Step 5: evaluate every model on the test set (reporting only)

KNN, the SVMs and the classifiers behind HOG and PCA were not saved (they would store the training data), so they are **refitted on the training split** from the settings saved in Phases 5 and 6, which takes seconds on the GPU. The refitted SVM can differ from the validation run in the last decimals (floating-point order on the GPU).
Each model is wrapped in `try/except`: a missing file prints a warning and the rest continues.

In [ ]:
cfg5 = json.loads((PROJECT_ROOT / "results/metrics/05_traditional_configs.json").read_text())
cfg6_path = PROJECT_ROOT / "results/metrics/06_feature_configs.json"
cfg6 = json.loads(cfg6_path.read_text()) if cfg6_path.exists() else None

def record(name, metrics, pred, probs, ms, notes=""):
    TEST[name] = dict(metrics=metrics, pred=np.asarray(pred), probs=np.asarray(probs), ms=ms)
    evaluation.log_result(name, "test", metrics, infer_ms_per_img=round(ms, 4), notes=notes)
    print(f"{name:34s} acc {metrics['accuracy']:.4f} | F1 {metrics['f1']:.4f} | AUC {metrics['roc_auc']:.4f} | {ms:.3f} ms/img")

def sync():
    if dev.type == "cuda": torch.cuda.synchronize()

def score_gpu(model, Xg):
    sync(); t0 = time.perf_counter(); s = model.score_dog(Xg); sync()
    ms = 1000 * (time.perf_counter() - t0) / len(Xg)
    return s, (s > model.threshold).astype(int), ms

def fit_gpu_clf(spec, Xg_tr, yg_tr):
    if spec["type"] == "lr":   m = gpu_models.GPULogisticRegression(spec["C"])
    elif spec["type"] == "knn": m = gpu_models.GPUKNN(k=spec["k"])
    else:                       m = gpu_models.GPURBFSVM(C=spec["C"], n_components=spec.get("D", 4096))
    return m.fit(Xg_tr, yg_tr)

# ---- raw-pixel (RGB 64x64) models ----
X64_tr = traditional_models.flatten(imgs[64][tr]); X64_te = traditional_models.flatten(test_imgs[64])
Xg_tr = torch.from_numpy(X64_tr).to(dev); yg_tr = torch.from_numpy(y_train).to(dev); Xg_te = torch.from_numpy(X64_te).to(dev)

try:
    maj = traditional_models.make_majority().fit(np.zeros((len(y_train), 1)), y_train)
    m, p, s, ms = evaluation.evaluate_sklearn(maj, np.zeros((len(y_test), 1)), y_test)
    record("Majority baseline", m, p, s, ms, "predicts the majority training class")
except Exception as e: print("Majority baseline FAILED:", e)

try:
    ck = torch.load(TRAD_DIR / "logreg_raw64_gpu.pt", map_location="cpu", weights_only=True)
    lr = gpu_models.GPULogisticRegression(float(ck["C"])); lr.scaler = gpu_models.GPUStandardScaler()
    lr.scaler.mean, lr.scaler.std, lr.w, lr.b = ck["mean"].to(dev), ck["std"].to(dev), ck["w"].to(dev), ck["b"].to(dev)
    s, p, ms = score_gpu(lr, Xg_te)
    record("Logistic Regression", evaluation.compute_metrics(y_test, p, s), p, s, ms, "loaded from saved weights")
except Exception as e: print("Logistic Regression FAILED:", e)

try:
    knn = fit_gpu_clf({"type": "knn", "k": cfg5["knn"]["k"]}, Xg_tr, yg_tr)
    s, p, ms = score_gpu(knn, Xg_te)
    record("KNN", evaluation.compute_metrics(y_test, p, s), p, s, ms, f"refit on train, k={cfg5['knn']['k']}")
except Exception as e: print("KNN FAILED:", e)

try:
    svm = fit_gpu_clf({"type": "svm", "C": cfg5["svm"]["C"], "D": cfg5["svm"]["n_components"]}, Xg_tr, yg_tr)
    s, p, ms = score_gpu(svm, Xg_te)
    record("SVM - Raw Pixels", evaluation.compute_metrics(y_test, p, s), p, s, ms, "approx RBF SVM, refit on train")
except Exception as e: print("SVM FAILED:", e)

try:
    rf = joblib.load(TRAD_DIR / "random_forest_raw64.joblib")
    m, p, s, ms = evaluation.evaluate_sklearn(rf, X64_te, y_test)
    record("Random Forest", m, p, s, ms, "loaded from joblib")
except Exception as e: print("Random Forest FAILED:", e)

In [ ]:
# ---- HOG + SVM and PCA + classifier (settings from Phase 6) ----
if cfg6 is None:
    print("06_feature_configs.json missing: skipping HOG and PCA rows (run Phase 6).")
else:
    try:
        hog_all = np.load(PROJECT_ROOT / "data/processed/hog_gray128.npy", mmap_mode="r")
        spec = {"type": "svm", "C": cfg6["hog"]["svm"]["C"], "D": cfg6["hog"]["svm"]["n_components"]}
        Hg_tr = torch.from_numpy(np.ascontiguousarray(hog_all[tr])).to(dev); Hg_te = torch.from_numpy(np.ascontiguousarray(hog_all[te])).to(dev)
        mdl = fit_gpu_clf(spec, Hg_tr, yg_tr); s, p, ms = score_gpu(mdl, Hg_te)
        record("HOG + SVM", evaluation.compute_metrics(y_test, p, s), p, s, ms, "approx RBF SVM on HOG, refit on train")
        del Hg_tr, Hg_te, mdl, hog_all; torch.cuda.empty_cache()
    except Exception as e: print("HOG + SVM FAILED:", e)

    try:
        pk = torch.load(TRAD_DIR / "pca_rgb64.pt", map_location="cpu", weights_only=True)
        mean_, comps = pk["mean"].to(dev), pk["components"].to(dev)
        Ztr, Zte = (Xg_tr - mean_) @ comps.T, (Xg_te - mean_) @ comps.T
        spec = cfg6["pca"]["classifier"]
        mdl = fit_gpu_clf(spec, Ztr, yg_tr); s, p, ms = score_gpu(mdl, Zte)
        name = cfg6["pca"]["best_classifier"]
        record(name, evaluation.compute_metrics(y_test, p, s), p, s, ms, f"{comps.shape[0]} PCs (fit on train), refit classifier")
        del Ztr, Zte, mdl; torch.cuda.empty_cache()
    except Exception as e: print("PCA + classifier FAILED:", e)

del Xg_tr, Xg_te, yg_tr; torch.cuda.empty_cache()

In [ ]:
# ---- deep models ----
best_cnn = json.loads((CNN_DIR / "best_custom_cnn.json").read_text())
mean128, std128 = np.array(best_cnn["mean"], np.float32), np.array(best_cnn["std"], np.float32)
mean224 = np.array([0.485, 0.456, 0.406], np.float32); std224 = np.array([0.229, 0.224, 0.225], np.float32)
PRETTY_TO_KEY = {v: k for k, v in pretrained_models.PRETTY.items()}

def deep_spec(name):
    if name.startswith("Custom CNN - "):
        arch = name.split(" - ", 1)[1]
        return dict(family="custom_cnn", arch_kwargs=dict(cnn_models.ARCHITECTURES[arch]), size=128, mean=mean128, std=std128,
                    ckpt=CNN_DIR / f"{arch.lower().replace('-', '_')}.pt")
    if name == "CNN + Augmentation":
        return dict(family="custom_cnn", arch_kwargs=dict(best_cnn["arch_kwargs"]), size=128, mean=mean128, std=std128,
                    ckpt=CNN_DIR / "cnn_aug.pt")
    if name.startswith("TL "):
        pretty, stage = name[3:name.rindex(" (")], name[name.rindex("(") + 1:-1]
        key = PRETTY_TO_KEY[pretty]
        spec = dict(family="transfer", backbone=key, size=224, mean=mean224, std=std224, dropout=0.3, stage=stage)
        spec["ckpt"] = TL_DIR / (f"{key}_finetuned.pt" if stage == "fine-tuned" else f"{key}_head_frozen.pt")
        return spec
    raise KeyError(name)

def build_deep(spec):
    if spec["family"] == "custom_cnn":
        m = cnn_models.CustomCNN(**spec["arch_kwargs"], input_size=spec["size"])
        m.load_state_dict(torch.load(spec["ckpt"], map_location="cpu", weights_only=True))
    elif spec["stage"] == "fine-tuned":
        m = pretrained_models.TransferModel(spec["backbone"], weights_path=None, dropout=spec["dropout"])
        m.load_state_dict(torch.load(spec["ckpt"], map_location="cpu", weights_only=True))
    else:        # frozen: ImageNet backbone + the trained head
        m = pretrained_models.TransferModel(spec["backbone"], weights_path=WEIGHTS_DIR / f"{spec['backbone']}_imagenet.pth", dropout=spec["dropout"])
        m.head[3].load_state_dict(torch.load(spec["ckpt"], map_location="cpu", weights_only=True))
    return m.eval()

DEEP_NAMES = list(deep["experiment"]) + [n for n in val["experiment"] if "control" in n.lower() and is_deep[val["experiment"] == n].any()]
DEEP_NAMES = [n for n in dict.fromkeys(DEEP_NAMES) if "control" not in n.lower()]       # the control is not evaluated
if FINAL_NAME not in DEEP_NAMES: DEEP_NAMES.append(FINAL_NAME)

for name in DEEP_NAMES:
    try:
        spec = deep_spec(name)
        model = build_deep(spec).to(dev).to(memory_format=torch.channels_last)
        metrics, pred, probs, ms = training.evaluate_dl(model, test_imgs[spec["size"]], y_test, spec["mean"], spec["std"], batch_size=128)
        record(name, metrics, pred, probs, ms, "loaded checkpoint (ms/img includes host-to-GPU copy)")
        del model; torch.cuda.empty_cache()
    except Exception as e:
        print(f"{name:34s} FAILED/skipped: {type(e).__name__}: {str(e)[:120]}")
assert FINAL_NAME in TEST, "The final model could not be evaluated: fix the error above first."

## Step 6: the consolidated comparison table (Q49)

The table reports **test** metrics, next to the validation accuracy (a large val-test gap would signal that validation tuning overfit), training time, inference time and parameters.
Check the DOCX's own list of required columns and rename or extend them if needed.

In [ ]:
res = pd.read_csv(evaluation.RESULTS_CSV)
vv = res[res["split"] == "val"].drop_duplicates("experiment", keep="last").set_index("experiment")
ORDER = ["Majority baseline", "Logistic Regression", "KNN", "SVM - Raw Pixels", "Random Forest", "HOG + SVM",
         "PCA + KNN", "PCA + LR", "PCA + SVM"] + [f"Custom CNN - {a}" for a in cnn_models.ARCHITECTURES] + ["CNN + Augmentation"]
names = [n for n in ORDER if n in TEST] + [n for n in TEST if n not in ORDER]
rows = []
for n in names:
    m = TEST[n]["metrics"]
    rows.append({"model": n, "test acc": m["accuracy"], "precision": m["precision"], "recall": m["recall"], "F1": m["f1"],
                 "ROC-AUC": m["roc_auc"], "val acc": vv.loc[n, "accuracy"] if n in vv.index else np.nan,
                 "train time (s)": vv.loc[n, "train_time_s"] if n in vv.index else np.nan,
                 "params": vv.loc[n, "params"] if n in vv.index and "params" in vv.columns else np.nan,
                 "ms/img (test)": TEST[n]["ms"]})
final_table = pd.DataFrame(rows)
final_table["val-test gap"] = final_table["val acc"] - final_table["test acc"]
pd.set_option("display.width", 220)
display(final_table.round(4))
final_table.to_csv(PROJECT_ROOT / "results/comparisons/final_comparison_test.csv", index=False)
try:
    final_table.to_excel(PROJECT_ROOT / "results/comparisons/final_comparison_test.xlsx", index=False)
except Exception as e:
    print("xlsx not written (openpyxl missing?):", e)
print("\nFINAL MODEL:", FINAL_NAME, "| test accuracy", round(TEST[FINAL_NAME]["metrics"]["accuracy"], 4))

In [ ]:
# ROC curves (test): non-trivial models; the final model is drawn thick
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for n in names:
    if n == "Majority baseline": continue
    fpr, tpr, _ = roc_curve(y_test, TEST[n]["probs"])
    lw = 3 if n == FINAL_NAME else 1
    for a in ax: a.plot(fpr, tpr, lw=lw, label=f"{n} ({TEST[n]['metrics']['roc_auc']:.3f})")
ax[0].plot([0, 1], [0, 1], "k:"); ax[0].set(xlabel="false positive rate", ylabel="true positive rate", title="ROC curves (test)")
ax[0].legend(fontsize=6, loc="lower right"); ax[1].set(xlim=(0, 0.15), ylim=(0.85, 1.0), title="zoom: top-left corner", xlabel="false positive rate")
plt.tight_layout(); plt.savefig(PROJECT_ROOT / "results/figures/09_roc_test.png", dpi=110); plt.show()

main = [n for n in ["Logistic Regression", "SVM - Raw Pixels", "Random Forest", "HOG + SVM"] + [x for x in names if x.startswith("PCA")]
        + [FINAL_NAME, "CNN + Augmentation"] if n in TEST]
main = list(dict.fromkeys(main))
fig, axes = plt.subplots(2, (len(main) + 1) // 2, figsize=(4 * ((len(main) + 1) // 2), 7.5))
for a, n in zip(axes.ravel(), main):
    evaluation.plot_confusion(y_test, TEST[n]["pred"], n, ax=a)
for a in axes.ravel()[len(main):]: a.axis("off")
plt.tight_layout(); plt.savefig(PROJECT_ROOT / "results/figures/09_confusion_test.png", dpi=110); plt.show()
print(f"Classification report - {FINAL_NAME} (test)")
print(classification_report(y_test, TEST[FINAL_NAME]["pred"], target_names=evaluation.CLASS_NAMES, digits=4))

## Step 7: overfitting and underfitting overview

`train acc` is measured in evaluation mode on a fixed subset of 3,000 training images, so a large gap to validation accuracy means memorisation. A low training accuracy *and* a low validation accuracy means underfitting (the raw-pixel models are the usual example).

In [ ]:
over = vv.loc[[n for n in names if n in vv.index], ["train_accuracy", "accuracy"]].rename(columns={"accuracy": "val acc"})
over["gap"] = over["train_accuracy"] - over["val acc"]
over["test acc"] = [TEST[n]["metrics"]["accuracy"] for n in over.index]
display(over.sort_values("gap", ascending=False).round(4))

## Step 8: error analysis of the final model (Q49)

Dog is the positive class: a **false positive** is a *cat predicted as dog*, a **false negative** is a *dog predicted as cat*.
We look at which images fail, how confident the model was, whether failures cluster in image properties (size, aspect ratio, brightness, file format), and whether other models fail on the same images.

In [ ]:
final = TEST[FINAL_NAME]
probs, pred = final["probs"], final["pred"]
wrong = pred != y_test
fp = (y_test == 0) & (pred == 1); fn = (y_test == 1) & (pred == 0)
n_cat, n_dog = int((y_test == 0).sum()), int((y_test == 1).sum())
print(f"{FINAL_NAME}: {wrong.sum()} errors out of {len(y_test)} ({wrong.mean():.2%})")
print(f"  false positives (cat -> dog): {fp.sum()}  = {fp.sum() / n_cat:.2%} of cats")
print(f"  false negatives (dog -> cat): {fn.sum()}  = {fn.sum() / n_dog:.2%} of dogs")

test_df["prob_dog"] = probs; test_df["pred"] = pred; test_df["wrong"] = wrong
test_df["confidence"] = np.maximum(probs, 1 - probs)
t64 = test_imgs[64].astype(np.float32) / 255.0
test_df["brightness"] = t64.mean(axis=(1, 2, 3)); test_df["contrast"] = t64.std(axis=(1, 2, 3))
test_df["min_side"] = test_df[["width", "height"]].min(axis=1); test_df["aspect"] = test_df["width"] / test_df["height"]
test_df["format"] = np.where(test_df["pil_format"] == "JPEG", "JPEG", "other (BMP/GIF/PNG/PSD)")

def err_table(col, bins=None, q=None):
    g = pd.cut(test_df[col], bins) if bins is not None else (pd.qcut(test_df[col], q) if q else test_df[col])
    t = test_df.groupby(g, observed=True)["wrong"].agg(n="size", errors="sum", error_rate="mean")
    return t
for title, t in [("image shortest side (px)", err_table("min_side", bins=[0, 150, 250, 350, 600])),
                 ("aspect ratio (w/h)", err_table("aspect", bins=[0, 0.8, 1.2, 1.6, 10])),
                 ("brightness quartile", err_table("brightness", q=4)),
                 ("contrast quartile", err_table("contrast", q=4)),
                 ("file format", err_table("format")), ("true class", err_table("label"))]:
    print(f"\nError rate by {title}  (groups with few images are unreliable)"); display(t.round(4))

In [ ]:
# The most confident mistakes (most informative), and the images the model was least sure about
def show(idx, title, cols=6, rows=2):
    fig, axes = plt.subplots(rows, cols, figsize=(2.6 * cols, 2.9 * rows))
    for a in axes.ravel(): a.axis("off")
    for a, i in zip(axes.ravel(), idx):
        a.imshow(test_imgs[128][i]); a.set_title(f"true {evaluation.CLASS_NAMES[y_test[i]]}\nP(dog)={probs[i]:.3f}", fontsize=8)
    fig.suptitle(title); plt.tight_layout(); return fig

fp_idx = np.where(fp)[0][np.argsort(-probs[fp])][:12]                  # cats the model was most sure were dogs
fn_idx = np.where(fn)[0][np.argsort(probs[fn])][:12]                   # dogs the model was most sure were cats
if len(fp_idx): show(fp_idx, "Most confident FALSE POSITIVES (cats predicted as dogs)").savefig(PROJECT_ROOT / "results/figures/09_false_positives.png", dpi=100); plt.show()
if len(fn_idx): show(fn_idx, "Most confident FALSE NEGATIVES (dogs predicted as cats)").savefig(PROJECT_ROOT / "results/figures/09_false_negatives.png", dpi=100); plt.show()
unsure_idx = np.argsort(np.abs(probs - 0.5))[:12]
show(unsure_idx, "The 12 least certain predictions (P close to 0.5)").savefig(PROJECT_ROOT / "results/figures/09_uncertain.png", dpi=100); plt.show()

errs = test_df.loc[wrong, ["path", "label", "prob_dog", "confidence", "width", "height", "pil_format", "brightness"]].sort_values("confidence", ascending=False)
errs.to_csv(PROJECT_ROOT / "results/metrics/09_test_errors.csv", index=False)
print("saved", len(errs), "misclassified test images -> results/metrics/09_test_errors.csv")

In [ ]:
# Are the hard images hard for every model?
deep_in_test = [n for n in DEEP_NAMES if n in TEST]
wrong_count = np.sum([TEST[n]["pred"] != y_test for n in deep_in_test], axis=0)
print(f"{len(deep_in_test)} deep models evaluated.")
print("images misclassified by ALL of them:", int((wrong_count == len(deep_in_test)).sum()),
      "| by at least half:", int((wrong_count >= len(deep_in_test) / 2).sum()),
      "| by the final model:", int(wrong.sum()))
hard = np.where(wrong_count == len(deep_in_test))[0][:12]
if len(hard):
    show(hard, "Misclassified by every deep model (likely ambiguous or mislabelled)").savefig(PROJECT_ROOT / "results/figures/09_hard_images.png", dpi=100); plt.show()

# Confidence behaviour: how much does the uncertainty threshold of the app buy us?
rows = []
for t in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]:
    keep = test_df["confidence"] >= t
    rows.append({"threshold": t, "coverage (share answered)": keep.mean(), "accuracy on answered": (~test_df.loc[keep, "wrong"]).mean(),
                 "images left 'uncertain'": int((~keep).sum())})
display(pd.DataFrame(rows).round(4))
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.histplot(data=test_df, x="confidence", hue=np.where(wrong, "wrong", "correct"), bins=30, multiple="stack", ax=ax)
ax.set_title("Confidence of correct vs wrong predictions (test)"); plt.tight_layout(); plt.show()

## Step 9: package the final model and test the deployment code (Q50)

The package folder `app_package/` holds exactly what the app needs: `model.pt` (state_dict), `model_config.json` (architecture, image size, normalisation, class mapping, threshold), `prediction.py`, `cnn_models.py`, `pretrained_models.py`, `streamlit_app.py` and `requirements_app.txt`. It is also zipped as `app_package.zip` for upload to Colab.

In [ ]:
PKG = PROJECT_ROOT / "app_package"
PKG.mkdir(exist_ok=True)
spec = deep_spec(FINAL_NAME)
model = build_deep(spec).cpu().eval()
torch.save(model.state_dict(), PKG / "model.pt")
cfg = {"final_model_name": FINAL_NAME, "family": spec["family"], "input_size": spec["size"],
       "mean": [float(v) for v in spec["mean"]], "std": [float(v) for v in spec["std"]],
       "class_names": evaluation.CLASS_NAMES, "label_map": {"Cat": 0, "Dog": 1}, "threshold": 0.5,
       "resize": "direct resize to a square (stretch), PIL bilinear; RGB; /255; normalised with mean/std",
       "test": {k: float(v) for k, v in TEST[FINAL_NAME]["metrics"].items()},
       "torch_version_at_export": torch.__version__}
if spec["family"] == "custom_cnn":
    cfg["arch_kwargs"] = {k: (list(v) if isinstance(v, tuple) else v) for k, v in spec["arch_kwargs"].items()}
else:
    cfg.update(backbone=spec["backbone"], dropout=spec["dropout"])
(PKG / "model_config.json").write_text(json.dumps(cfg, indent=2))
for f in ["prediction.py", "cnn_models.py", "pretrained_models.py"]:
    shutil.copy(PROJECT_ROOT / "src" / f, PKG / f)
shutil.copy(PROJECT_ROOT / "app/streamlit_app.py", PKG / "streamlit_app.py")
(PKG / "requirements_app.txt").write_text("streamlit\ntorch\ntorchvision\npillow\nnumpy\n")
zip_path = shutil.make_archive(str(PROJECT_ROOT / "app_package"), "zip", PKG)
for p in sorted(PKG.iterdir()):
    print(f"{p.stat().st_size / 1e6:8.2f} MB  {p.name}")
print(f"\nzip: {zip_path}  ({Path(zip_path).stat().st_size / 1e6:.1f} MB)")

In [ ]:
# Test the packaged model exactly as the app will use it (CPU, one image at a time, original image files)
sys.path.insert(0, str(PKG))
import prediction
importlib.reload(prediction)
predictor = prediction.load_predictor(PKG, device=torch.device("cpu"))

n_chk = 30
rows = []
for i in range(n_chk):
    r = predictor.predict(test_df.loc[i, "path"])
    rows.append({"true": test_df.loc[i, "label"], "app label": r["label"], "app P(dog)": r["p_dog"],
                 "batch P(dog)": float(probs[i]) if FINAL_NAME == FINAL_NAME else np.nan, "ms": r["ms"]})
chk = pd.DataFrame(rows)
chk["abs diff"] = (chk["app P(dog)"] - chk["batch P(dog)"]).abs()
chk["same label"] = chk["app label"] == np.where(chk["batch P(dog)"] >= 0.5, "Dog", "Cat")
display(chk.head(8).round(4))
print(f"labels identical to the evaluation pipeline: {chk['same label'].mean():.0%} | max |P(dog) difference|: {chk['abs diff'].max():.4f} "
      f"(small differences are expected: the evaluation used bfloat16 on the GPU, the app test used float32 on the CPU)")
print(f"CPU latency per image, including decoding and resizing: median {chk['ms'].median():.1f} ms, mean {chk['ms'].mean():.1f} ms")

In [ ]:
# Invalid inputs must fail gracefully (the app shows these messages instead of crashing)
import io
from PIL import Image
buf = io.BytesIO(); Image.new("RGB", (10, 10), "white").save(buf, format="PNG")
cases = {"plain text": b"this is not an image", "empty file": b"", "tiny 10x10 image": buf.getvalue(),
         "truncated JPEG": open(test_df.loc[0, "path"], "rb").read()[:300]}
for label, data in cases.items():
    try:
        r = predictor.predict(data); print(f"{label:18s} -> predicted {r['label']} (unexpected: check the input)")
    except prediction.InvalidImageError as e:
        print(f"{label:18s} -> rejected: {e}")

# A few challenging examples through the app code path: the least certain and the most confident wrong test images
for i in list(unsure_idx[:3]) + list(fp_idx[:1]) + list(fn_idx[:1]):
    r = predictor.predict(test_df.loc[i, "path"])
    print(f"true {test_df.loc[i, 'label']:3s} | app says {r['label']} ({r['confidence']:.1%}) | {Path(test_df.loc[i, 'path']).name}")

## Step 10: README and reproducibility files

The README is generated from the actual results (final model name and test metrics come from the code above, nothing is typed in by hand). Review it and add your name, course and any extra notes.

In [ ]:
m = TEST[FINAL_NAME]["metrics"]
table_md = ["| Model | Test acc | F1 | ROC-AUC |", "|---|---|---|---|"] + \
           [f"| {r['model']} | {r['test acc']:.4f} | {r['F1']:.4f} | {r['ROC-AUC']:.4f} |" for _, r in final_table.iterrows()]
readme = f"""# Cat vs Dog Image Classification (SIC AI Capstone)

Binary image classification (Cat / Dog) comparing traditional ML (raw pixels, HOG, PCA), custom CNNs, data augmentation and
transfer learning (MobileNetV2, ResNet50, VGG16, EfficientNet-B0), with a Streamlit demo.

**Final model (chosen on validation data):** {FINAL_NAME}
**Held-out test results:** accuracy {m['accuracy']:.4f}, precision {m['precision']:.4f}, recall {m['recall']:.4f}, F1 {m['f1']:.4f}, ROC-AUC {m['roc_auc']:.4f}

## Results (test split, {len(y_test)} images)
{chr(10).join(table_md)}

Note: the SVM rows use a random-Fourier-feature approximation of an RBF SVM trained on the GPU, not exact libsvm.

## Project structure
- `notebooks/` 01-09 (one per phase), in order
- `src/` reusable modules: data_processing, evaluation, traditional_models, gpu_models, feature_extraction, experiments, cnn_models, training, augmentation, pretrained_models, prediction
- `data/splits/split_seed42.csv` fixed train/val/test split (70/15/15, stratified, near-duplicate groups kept together, seed 42)
- `results/` metrics, comparison tables and figures; `models/` trained weights (not in Git); `app/`, `app_package/` the Streamlit demo

## Reproduce
1. Environment: Python 3.10, PyTorch 2.5 (NVIDIA container 24.10), see `requirements.txt`.
2. Put the extracted dataset at `Cats and Dogs Classification Dataset/{{Cat,Dog}}` (it is not in Git).
3. Run notebooks 02 to 09 in order (04 builds the cached arrays and the split; 06-08 train the models; 09 evaluates on the test split once).
4. ImageNet weights (`models/pretrained/*.pth`) were exported from torchvision on a machine with internet access and uploaded (the training server is offline).
5. Seeds are fixed (42); GPU kernels are not fully deterministic, so tiny run-to-run differences are possible.

## Run the app
Local (needs Python, `pip install -r app_package/requirements_app.txt`):
`cd app_package && streamlit run streamlit_app.py`
Colab: open `notebooks/09b_colab_streamlit_demo.ipynb` and upload `app_package.zip`.

## Limitations
Binary task on one dataset; images were stretched to a square; hyperparameters were tuned on the validation split; each model was trained
with a single seed; the app is a demo, not a production service; photos of other animals will still be forced into Cat or Dog
(use the uncertainty slider as a partial safeguard).
"""
(PROJECT_ROOT / "README.md").write_text(readme)
req = PROJECT_ROOT / "requirements.txt"
if not req.exists():
    req.write_text("numpy==1.24.4\npandas==2.2.2\nmatplotlib==3.9.2\nseaborn==0.13.2\nscikit-learn==1.5.2\nscikit-image==0.25.2\n"
                   "opencv-python==4.7.0\npillow==10.4.0\nimagehash==4.3.2\njoblib==1.4.2\ntqdm==4.66.5\n"
                   "torch   # NVIDIA PyTorch container 24.10 (2.5.0a0)\ntorchvision   # 0.20.0a0\n")
print("README.md written. Preview:\n"); print(readme[:1500])

In [ ]:
# Deliverables audit
checks = {"notebooks 01-09": [f"notebooks/{n}" for n in ["01_problem_understanding.ipynb", "02_eda.ipynb", "03_data_cleaning.ipynb", "04_preprocessing.ipynb",
          "05_traditional_ml.ipynb", "06_hog_pca.ipynb", "07_custom_cnn.ipynb", "08_augmentation_transfer_learning.ipynb", "09_evaluation_deployment.ipynb"]],
          "source files": [f"src/{n}.py" for n in ["data_processing", "evaluation", "traditional_models", "gpu_models", "feature_extraction", "experiments",
                                                    "cnn_models", "training", "augmentation", "pretrained_models", "prediction"]],
          "models": ["models/traditional/random_forest_raw64.joblib", "models/traditional/pca_rgb64.pt", "models/cnn/best_custom_cnn.json",
                     "models/cnn/cnn_aug.pt", "models/transfer/best_transfer.json", "app_package/model.pt", "app_package/model_config.json"],
          "app and docs": ["app/streamlit_app.py", "app_package.zip", "README.md", "requirements.txt",
                           "results/comparisons/model_results.csv", "results/comparisons/final_comparison_test.csv"],
          "still to write": ["reports/project_report.docx", "reports/presentation.pptx"]}
for group, files in checks.items():
    print(f"\n{group}")
    for f in files:
        print("  ", "OK     " if (PROJECT_ROOT / f).exists() else "MISSING", f)

## Your write-ups (use your own numbers and words)

**Q49.** Report the final model's accuracy, precision, recall, F1 and AUC on the test split; describe the confusion matrix (how many cats called dogs, how many dogs called cats, and which is the weaker class); compare it with the other models using the consolidated table, and say whether the val-test gap suggests any overfitting to the validation set. From the error analysis, describe the *types* of images that fail most (look at the confident false positives/negatives, the bins by size/aspect/brightness/format, and the images misclassified by every deep model: ambiguous, partially visible animals, unusual poses, possible label noise?). Say what you would do about them (more data, targeted augmentation, higher resolution, relabelling).

**Final model justification.** Explain the selection rule (best validation accuracy, within 0.3 pp prefer fewer parameters), the trade-off between accuracy, size and speed (use `params`, `ms/img`, CPU latency), and why accuracy alone was not the criterion.

**Q50.** Describe the app (upload, preview, class, confidence, probabilities, uncertainty slider, error handling), how it loads the saved model rather than retraining, and what you tested (the 30-image parity check, the invalid inputs, the difficult examples). Add screenshots from the Colab run.

**Honest limitations for the report:** single seed per model; SVM is an approximation; hyperparameters tuned on validation data; images stretched to square; the app forces every photo into Cat or Dog.

**Still to do:** run `09b_colab_streamlit_demo.ipynb` on Colab, then the project **report** and **presentation** (they must quote only the real results above).